# EDA and cleaning

This notebook documents the full cleaning pipeline.

## 0. Data retrieval


In [1]:
import os

import numpy as np
import pandas as pd

data_path = "../data/regularite-mensuelle-tgv-aqst.csv"

df = pd.read_csv(data_path, sep=";")

print("Dimensions:", df.shape)
df.head(3)

Dimensions: (12907, 26)


,date,service,gare_depart,gare_arrivee,duree_moyenne,nb_train_prevu,nb_annulation,commentaire_annulation,nb_train_depart_retard,retard_moyen_depart,...,nb_train_retard_sup_15,retard_moyen_trains_retard_sup15,nb_train_retard_sup_30,nb_train_retard_sup_60,prct_cause_externe,prct_cause_infra,prct_cause_gestion_trafic,prct_cause_materiel_roulant,prct_cause_gestion_gare,prct_cause_prise_en_charge_voyageurs
0,2018-01,National,BORDEAUX ST JEAN,PARIS MONTPARNASSE,141,870,5,NaN,289,11.247809,...,110,6.511118,44,8,36.134454,31.092437,10.924370,15.966387,5.042017,0.840336
1,2018-01,National,LE MANS,PARIS MONTPARNASSE,56,406,1,NaN,213,8.479969,...,32,5.363539,9,4,20.000000,35.000000,16.666667,16.666667,8.333333,3.333333
2,2018-01,National,PARIS MONTPARNASSE,LA ROCHELLE VILLE,166,226,0,NaN,21,6.239683,...,11,2.938053,6,1,22.222222,27.777778,16.666667,16.666667,5.555556,11.111111


## 1. First inspection: structure and types

In [2]:
print("Period:", df["date"].min(), "to", df["date"].max())
print("Number of unique departure stations:", df["gare_depart"].nunique())
print("Number of unique routes:", df.groupby(["gare_depart", "gare_arrivee"]).ngroups)
print("Services:", df["service"].unique().tolist())
print("")  # noqa: FURB105
print("type by columns")
df.dtypes

Period: 2018-01 to 2026-09
Number of unique departure stations: 59
Number of unique routes: 130
Services: ['National', 'International']

type by columns


date                                        str
service                                     str
gare_depart                                 str
gare_arrivee                                str
duree_moyenne                             int64
nb_train_prevu                            int64
nb_annulation                             int64
commentaire_annulation                  float64
nb_train_depart_retard                    int64
retard_moyen_depart                     float64
retard_moyen_tous_trains_depart         float64
commentaire_retards_depart              float64
nb_train_retard_arrivee                   int64
retard_moyen_arrivee                    float64
retard_moyen_tous_trains_arrivee        float64
commentaires_retard_arrivee                 str
nb_train_retard_sup_15                    int64
retard_moyen_trains_retard_sup15        float64
nb_train_retard_sup_30                    int64
nb_train_retard_sup_60                    int64
prct_cause_externe                      

In [3]:
pct_empty = df.isna().mean()
pct_empty[pct_empty == 1]

commentaire_annulation        1.0
commentaire_retards_depart    1.0
dtype: float64

## 2. Initial cleaning: dropping the empty columns

`commentaire_annulation` and `commentaire_retards_depart` are 100% empty:
they carry no information. We drop them. The third comment column contains
698 real texts, we keep it.

In [4]:
df = df.drop(columns=["commentaire_annulation", "commentaire_retards_depart"])

df["commentaires_retard_arrivee"].notna().sum()

np.int64(698)

## 3. Date conversion and chronological sort

The date column is read as text ("2018-01"). We convert it to datetime and we sort the dataset chronologicaly.

In [5]:
df["date"] = pd.to_datetime(df["date"], format="%Y-%m")
df = df.sort_values("date").reset_index(drop=True)

## 4. Mapping the missing values

by columns

In [6]:
df.isna().sum()

date                                        0
service                                     0
gare_depart                                 0
gare_arrivee                                0
duree_moyenne                               0
nb_train_prevu                              0
nb_annulation                               0
nb_train_depart_retard                      0
retard_moyen_depart                         0
retard_moyen_tous_trains_depart             0
nb_train_retard_arrivee                     0
retard_moyen_arrivee                        0
retard_moyen_tous_trains_arrivee            0
commentaires_retard_arrivee             12209
nb_train_retard_sup_15                      0
retard_moyen_trains_retard_sup15            0
nb_train_retard_sup_30                      0
nb_train_retard_sup_60                      0
prct_cause_externe                          0
prct_cause_infra                            0
prct_cause_gestion_trafic                   0
prct_cause_materiel_roulant       

In [7]:
missing_summary = pd.DataFrame(
    {
        "n_missing": df.isna().sum(),
        "percentage": (df.isna().mean() * 100).round(2),
    }
)
print(
    "Complete columns:",
    (missing_summary["n_missing"] == 0).sum(),
    "/",
    len(missing_summary),
)
missing_summary[missing_summary["n_missing"] > 0]

Complete columns: 23 / 24


,n_missing,percentage
commentaires_retard_arrivee,12209,94.59


by rows

In [8]:
missing_per_row = df.isna().sum(axis=1)
missing_per_row.value_counts().sort_index()

0      698
1    12209
Name: count, dtype: int64

In [9]:
col_miss_val = [c for c in df.columns if df[c].isna().any()]
print("Columns with missing values:", col_miss_val)

if col_miss_val:
    print(df.isna()[col_miss_val].value_counts())

Columns with missing values: ['commentaires_retard_arrivee']
commentaires_retard_arrivee
True                           12209
False                            698
Name: count, dtype: int64


In [10]:
years = df["date"].dt.year
has_comment = df["commentaires_retard_arrivee"].notna()
n_rows = years.value_counts().sort_index()
com_present = has_comment.groupby(years).sum().astype(int)
comments_by_year = pd.DataFrame({"present": com_present})
comments_by_year

,present
date,
2018,457
2019,241
2020,0
2021,0
2022,0
2023,0
2024,0
2025,0
2026,0


No row has more than one missing value. They are from the comments column. Comments concentrate on 2018-2019 period then disappear entirely from 2020 onwards. This absence of data is this feature is a Misssing Not At Random (MNAR) since this column stop beeing filed after 2020.

## 5. Duplicates

We realize that when look out for duplicates, we found some when we were strictly seeing date, leaving and arriving stations. Which should be not possible

In [11]:
cols = ["date", "gare_depart", "gare_arrivee"]
print("Duplicates on date + route:", df.duplicated(subset=cols).sum())
print("Total rows involved:", df.duplicated(subset=cols, keep=False).sum())

Duplicates on date + route: 42
Total rows involved: 84


The most likely explanation lies in the “service” column, which distinguishes between domestic and international trains. Initially, trains with the “international” service designation referred to departures or arrivals at stations outside mainland France. Here, these duplicates only concern stations within France. This means that the “service” label was used incorrectly


In [12]:
pair_rows = df[df.duplicated(subset=cols, keep=False)]
n_pairs = (pair_rows.groupby(cols)["service"].nunique() == 2).sum()
print("National/International pairs:", n_pairs, "/", pair_rows.groupby(cols).ngroups)
print("Months covered:", sorted(pair_rows["date"].dt.strftime("%Y-%m").unique()))

pair_rows[["date", "service", "gare_depart", "gare_arrivee"]].head(6)

National/International pairs: 42 / 42
Months covered: ['2025-07', '2025-08', '2025-09']


,date,service,gare_depart,gare_arrivee
11050,2025-07-01,National,CHAMBERY CHALLES LES EAUX,PARIS LYON
11052,2025-07-01,National,DIJON VILLE,PARIS LYON
11061,2025-07-01,National,PARIS LYON,DIJON VILLE
11066,2025-07-01,International,PARIS LYON,MULHOUSE VILLE
11069,2025-07-01,International,BELLEGARDE (AIN),PARIS LYON
11095,2025-07-01,National,LYON PART DIEU,MARSEILLE ST CHARLES


We will change this service to “National” and adjust the averages as needed.

When metrics count things (e.g. nb_train_prevu) we sum. When metrics mean things we use a weighted average

In [13]:
n_services = df.groupby(cols)["service"].transform("nunique")
to_merge = df[n_services == 2]
to_keep = df[n_services == 1]

counters = [
    "nb_train_prevu",
    "nb_annulation",
    "nb_train_depart_retard",
    "nb_train_retard_arrivee",
    "nb_train_retard_sup_15",
    "nb_train_retard_sup_30",
    "nb_train_retard_sup_60",
]

weighted = {
    "duree_moyenne": "nb_train_prevu",
    "retard_moyen_depart": "nb_train_depart_retard",
    "retard_moyen_tous_trains_depart": "nb_train_prevu",
    "retard_moyen_arrivee": "nb_train_retard_arrivee",
    "retard_moyen_tous_trains_arrivee": "nb_train_prevu",
    "retard_moyen_trains_retard_sup15": "nb_train_retard_sup_15",
    "prct_cause_externe": "nb_train_retard_arrivee",
    "prct_cause_infra": "nb_train_retard_arrivee",
    "prct_cause_gestion_trafic": "nb_train_retard_arrivee",
    "prct_cause_materiel_roulant": "nb_train_retard_arrivee",
    "prct_cause_gestion_gare": "nb_train_retard_arrivee",
    "prct_cause_prise_en_charge_voyageurs": "nb_train_retard_arrivee",
}

all_cols = list(df.columns)


def merge_pair(g):
    """g contains exactly 2 rows: National and International."""
    row = {}
    for c in cols:
        row[c] = g[c].iloc[0]
    row["service"] = "National"  # We choose to replace International by National
    for c in counters:
        row[c] = g[c].sum()
    for c, w in weighted.items():
        vals = g[c].to_numpy(dtype=float)
        weights = g[w].to_numpy(dtype=float)
        if weights.sum() > 0:
            row[c] = float((vals * weights).sum() / weights.sum())
        else:
            row[c] = float(vals.mean())
    comments = g["commentaires_retard_arrivee"].dropna().tolist()
    row["commentaires_retard_arrivee"] = " | ".join(comments) if comments else pd.NA
    return pd.Series([row.get(c) for c in all_cols], index=all_cols)


merged_rows = pd.DataFrame([merge_pair(g) for _, g in to_merge.groupby(cols)])
df = pd.concat([to_keep, merged_rows], ignore_index=True)
df = df.sort_values(["date", "gare_depart", "gare_arrivee"]).reset_index(drop=True)

print("Duplicates on date + route:", df.duplicated(subset=cols).sum())

Duplicates on date + route: 0


## 6. Check consistency in metrics

Descriptive comments are provided for each metric in the following cell.

In [14]:
trains_operated = df["nb_train_prevu"] - df["nb_annulation"]

mean_delays = [
    "retard_moyen_depart",
    "retard_moyen_tous_trains_depart",
    "retard_moyen_arrivee",
    "retard_moyen_tous_trains_arrivee",
    "retard_moyen_trains_retard_sup15",
]
dep_delayed = df["nb_train_depart_retard"] > 0    # trains that departed late
arr_delayed = df["nb_train_retard_arrivee"] > 0   # trains that arrived late
mask_zero_delayed_means = (dep_delayed & (df["retard_moyen_depart"] == 0)) | (arr_delayed & (df["retard_moyen_arrivee"] == 0))                            # a mean delay of delayed trains cannot be 0 when trains were delayed
mask_zero_all_train_means = (dep_delayed & (df["retard_moyen_tous_trains_depart"] == 0)) | (arr_delayed & (df["retard_moyen_tous_trains_arrivee"] == 0))  # same constraint on the all-train means

checks = {"negative counter": (df[counters] < 0).any(axis=1),                                          # Their should be no negative counter, since it can only adds up
    "Zero scheduled trains": df["nb_train_prevu"]== 0,                                                 # A month without scheduled train service shouldn't be possible
    "Cancellations > scheduled trains": df["nb_annulation"] > df["nb_train_prevu"],                    # The number of cancellations is not expected to exceed the number of trains scheduled for the day
    "Departure delays > operated trains": df["nb_train_depart_retard"] > trains_operated,              # The number of trains that depart late must not be strictly greater than the scheduled number of trains.
    "Arrival delays > operated trains": df["nb_train_retard_arrivee"] > trains_operated,               # Same as above but with arrival delays
    "(Delays >15min) > arrival delays": df["nb_train_retard_sup_15"] > df["nb_train_retard_arrivee"],  # For trains with + 15min delay
    "(Delays >30min) > delays >15min": df["nb_train_retard_sup_30"] > df["nb_train_retard_sup_15"],    # 30 min
    "(Delays >60min) > delays >30min": df["nb_train_retard_sup_60"] > df["nb_train_retard_sup_30"],    # 60 min
    "Zero or negative average duration": df["duree_moyenne"] <= 0,                                     # duration should not be 0 or negative
    "Negative average delays": (df[mean_delays] < 0).any(axis=1),                                      # An average delay cannot be negative
    "Zero delayed-train average with delays": mask_zero_delayed_means,                                 # A mean delay of delayed trains cannot be 0 or lower when trains were delayed
    "Zero all-trains average with delays": mask_zero_all_train_means,                                  # Same constraint on the all-train means
    "(Mean delay >15min) < 15min": df["retard_moyen_trains_retard_sup15"] < 15,                        # The mean delay of trains delayed > 15 min cannot be below 15 min
}

results = pd.DataFrame(
    {
        "errors": [int(m.sum()) for m in checks.values()],
    },
    index=list(checks.keys()),
)
results

,errors
negative counter,43
Zero scheduled trains,73
Cancellations > scheduled trains,63
Departure delays > operated trains,64
Arrival delays > operated trains,63
(Delays >15min) > arrival delays,31
(Delays >30min) > delays >15min,3
(Delays >60min) > delays >30min,354
Zero or negative average duration,73
Negative average delays,277


In [15]:
years = df["date"].dt.year
anomalies_by_year = pd.DataFrame(
    {name: mask.groupby(years).sum().astype(int) for name, mask in checks.items()},
    index=sorted(years.unique()),
)
anomalies_by_year["TOTAL"] = anomalies_by_year.sum(axis=1)
anomalies_by_year.loc["TOTAL"] = anomalies_by_year.sum()
anomalies_by_year

,negative counter,Zero scheduled trains,Cancellations > scheduled trains,Departure delays > operated trains,Arrival delays > operated trains,(Delays >15min) > arrival delays,(Delays >30min) > delays >15min,(Delays >60min) > delays >30min,Zero or negative average duration,Negative average delays,Zero delayed-train average with delays,Zero all-trains average with delays,(Mean delay >15min) < 15min,TOTAL
2018,0,0,0,0,0,6,0,0,0,22,0,0,1481,1509
2019,0,0,0,0,0,7,0,0,0,31,1,0,494,533
2020,0,73,63,63,63,18,2,0,73,81,0,0,121,557
2021,0,0,0,0,0,0,0,0,0,82,17,1,28,128
2022,0,0,0,1,0,0,0,0,0,23,10,0,19,53
2023,0,0,0,0,0,0,0,0,0,8,3,0,5,16
2024,0,0,0,0,0,0,0,0,0,13,3,0,18,34
2025,43,0,0,0,0,0,1,354,0,13,4,0,7,422
2026,0,0,0,0,0,0,0,0,0,4,0,0,1,5
TOTAL,43,73,63,64,63,31,3,354,73,277,38,1,2174,3257


In [16]:
for name, mask in checks.items():
    n = int(mask.sum())
    print("#####")
    print(f"{name}: {n} rows")
    if n:
        print(
            "Months involved:",
            sorted(df.loc[mask, "date"].dt.strftime("%Y-%m").unique().tolist()),
        )

#####
negative counter: 43 rows
Months involved: ['2025-01', '2025-02', '2025-03']
#####
Zero scheduled trains: 73 rows
Months involved: ['2020-04', '2020-05', '2020-06', '2020-08', '2020-09', '2020-10', '2020-11', '2020-12']
#####
Cancellations > scheduled trains: 63 rows
Months involved: ['2020-04', '2020-05', '2020-06']
#####
Departure delays > operated trains: 64 rows
Months involved: ['2020-04', '2020-05', '2020-06', '2022-06']
#####
Arrival delays > operated trains: 63 rows
Months involved: ['2020-04', '2020-05', '2020-06']
#####
(Delays >15min) > arrival delays: 31 rows
Months involved: ['2018-01', '2018-02', '2018-03', '2019-05', '2019-12', '2020-04', '2020-05', '2020-06', '2020-07', '2020-08', '2020-09', '2020-10', '2020-11']
#####
(Delays >30min) > delays >15min: 3 rows
Months involved: ['2020-05', '2025-03']
#####
(Delays >60min) > delays >30min: 354 rows
Months involved: ['2025-01', '2025-02', '2025-03']
#####
Zero or negative average duration: 73 rows
Months involved: ['20

We have observed that the dataset contains inconsistencies and anomalies. We will go through them period by period. 

### 6.1 Covid Period

During 2020, there were many anomalies and inconsistencies, particularly from April 2020 to December 2020. This period reflects the Covid restrictions in France at that time, during which train traffic was heavily disrupted. These are therefore not anomalies to be corrected. We therefore decide to leave them as they are in the dataset.

### 6.2 Pre-Covid anomalies

We observed 13 anomalies during the pre-Covid period. These anomalies mainly stem from an inconsistency between the number of trains delayed by more than 15 minutes and the total number of trains.

In [17]:
time_threshold = pd.Timestamp("2020-01")  # first lockdown

mask_precovid = (df["nb_train_retard_sup_15"] > df["nb_train_retard_arrivee"]) & (
    df["date"] < time_threshold
)
anomalies_precovid = df[mask_precovid]
anomalies_precovid[
    [
        "date",
        "gare_depart",
        "gare_arrivee",
        "nb_train_retard_arrivee",
        "nb_train_retard_sup_15",
        "nb_train_retard_sup_30",
        "nb_train_retard_sup_60",
    ]
]

,date,gare_depart,gare_arrivee,nb_train_retard_arrivee,nb_train_retard_sup_15,nb_train_retard_sup_30,nb_train_retard_sup_60
19,2018-01-01,GRENOBLE,PARIS LYON,23,25,13,6
22,2018-01-01,LAUSANNE,PARIS LYON,21,22,9,2
149,2018-02-01,GRENOBLE,PARIS LYON,35,41,18,2
187,2018-02-01,PARIS EST,NANCY,38,44,16,4
279,2018-03-01,GRENOBLE,PARIS LYON,31,32,9,3
330,2018-03-01,PARIS LYON,GRENOBLE,37,38,21,5
2058,2019-05-01,BARCELONA,PARIS LYON,7,13,5,3
2114,2019-05-01,PARIS LYON,BARCELONA,7,12,7,2
2929,2019-12-01,FRANCFORT,PARIS EST,13,18,10,2
2970,2019-12-01,PARIS EST,STRASBOURG,23,26,8,0


We decided to apply an adjustment so that the total number of delayed trains would not be less than the number of trains delayed by more than 15 minutes. This makes these instances more consistent.  

### 6.3 Post-COVID anomalies

Over the post-Covid period, we noticed a much larger number of anomalies. First, we can see that there is one item where the number of trains delayed by more than 30 minutes exceeded the number of trains delayed by more than 15 minutes in March 2025.

In [18]:
# Post-COVID anomaly, 2025: more trains delayed > 30 min than > 15 min.
mask_postcovid1 = (
    (df["nb_train_retard_sup_30"] > df["nb_train_retard_sup_15"])
    & (df["date"] >= pd.Timestamp("2025-01"))
)

anomalies_sub15 = df[mask_postcovid1]

anomalies_sub15[
    [
        "date",
        "gare_depart",
        "gare_arrivee",
        "nb_train_retard_arrivee",
        "nb_train_retard_sup_15",
        "nb_train_retard_sup_30",
        "nb_train_retard_sup_60",
    ]
]

,date,gare_depart,gare_arrivee,nb_train_retard_arrivee,nb_train_retard_sup_15,nb_train_retard_sup_30,nb_train_retard_sup_60
10683,2025-03-01,TOURS,PARIS MONTPARNASSE,14,2,6,2


Secondly, we also observe many anomalies in the number of trains delayed by more than 60 minutes relative to the number of trains delayed by more than 30 minutes. We find 354 items with this anomaly in 2025.

In [19]:
mask_postcovid2 = df["nb_train_retard_sup_60"] > df["nb_train_retard_sup_30"]

anomalies_postcovid2 = df[mask_postcovid2]

anomalies_postcovid2.head(12)[["date",
                               "gare_depart",
                               "gare_arrivee",
                               "nb_train_retard_arrivee",
                               "nb_train_retard_sup_15",
                               "nb_train_retard_sup_30",
                               "nb_train_retard_sup_60"
                               ]]

,date,gare_depart,gare_arrivee,nb_train_retard_arrivee,nb_train_retard_sup_15,nb_train_retard_sup_30,nb_train_retard_sup_60
10324,2025-01-01,AIX EN PROVENCE TGV,PARIS LYON,60,57,0,31
10325,2025-01-01,ANGERS SAINT LAUD,PARIS MONTPARNASSE,96,49,-12,28
10326,2025-01-01,ANGOULEME,PARIS MONTPARNASSE,44,28,0,10
10327,2025-01-01,ANNECY,PARIS LYON,17,17,0,7
10328,2025-01-01,ARRAS,PARIS NORD,136,58,0,22
10329,2025-01-01,AVIGNON TGV,PARIS LYON,65,48,-5,25
10330,2025-01-01,BARCELONA,PARIS LYON,10,10,0,7
10331,2025-01-01,BELLEGARDE (AIN),PARIS LYON,36,27,0,19
10332,2025-01-01,BESANCON FRANCHE COMTE TGV,PARIS LYON,24,20,0,14
10333,2025-01-01,BORDEAUX ST JEAN,PARIS MONTPARNASSE,180,132,0,69


Unfortunately, those data are beyond recovery. Since the "number of delayed trains" metric encompasses all the metrics for the number of trains delayed beyond a given time threshold, it is not possible to reconstruct or correct this particular metric. And given the number of items to correct, doing so would risk introducing a significant bias into our analysis. We therefore prefer to replace these values with NaN values.

In [20]:
# For the First case 
df.loc[mask_postcovid1, "nb_train_retard_sup_15"] = pd.NA
# For the second case 
df.loc[mask_postcovid2, "nb_train_retard_sup_30"] = pd.NA

### 6.4 Cause percentages

We now focus on the inconsistency and anomalies with the causes of delays. The six `prct_cause_*` columns break the delayed trains down by cause. Their sum must be 100% on every row with at least one delayed train.


In [21]:
prct_causes = [
    "prct_cause_externe",
    "prct_cause_infra",
    "prct_cause_gestion_trafic",
    "prct_cause_materiel_roulant",
    "prct_cause_gestion_gare",
    "prct_cause_prise_en_charge_voyageurs",
]

cause_sum = df[prct_causes].sum(axis=1)
delayed = df["nb_train_retard_arrivee"]

complete = cause_sum.between(99.9, 100.1)
zero_no_delay = (cause_sum == 0) & (delayed == 0)
zero_with_delay = (cause_sum == 0) & (delayed > 0)
partial = (cause_sum > 0) & ~complete

print("Rows summing to 100 (+- 0.1 pt):", int(complete.sum()), "/", len(df))
print("Rows at 0 with no delayed train (coherent):", int(zero_no_delay.sum()))
print("Rows at 0 with delayed trains (no breakdown):", int(zero_with_delay.sum()))
print("Rows with an incomplete breakdown:", int(partial.sum()))

df.loc[
    zero_with_delay,
    ["date", "gare_depart", "gare_arrivee", "nb_train_retard_arrivee"] + prct_causes,
].head(5)


Rows summing to 100 (+- 0.1 pt): 12584 / 12865
Rows at 0 with no delayed train (coherent): 191
Rows at 0 with delayed trains (no breakdown): 80
Rows with an incomplete breakdown: 10


,date,gare_depart,gare_arrivee,nb_train_retard_arrivee,prct_cause_externe,prct_cause_infra,prct_cause_gestion_trafic,prct_cause_materiel_roulant,prct_cause_gestion_gare,prct_cause_prise_en_charge_voyageurs
301,2018-03-01,MARSEILLE ST CHARLES,MADRID,1,0.0,0.0,0.0,0.0,0.0,0.0
526,2018-05-01,BARCELONA,PARIS LYON,12,0.0,0.0,0.0,0.0,0.0,0.0
527,2018-05-01,BELLEGARDE (AIN),PARIS LYON,45,0.0,0.0,0.0,0.0,0.0,0.0
537,2018-05-01,FRANCFORT,PARIS EST,38,0.0,0.0,0.0,0.0,0.0,0.0
538,2018-05-01,GENEVE,PARIS LYON,43,0.0,0.0,0.0,0.0,0.0,0.0


The rows at 0 with no delayed train are coherent. They stay as they are.
There are two types of anomalies. Indices that have delayed trains but for which the percentage of causes is 0, and those where the sum does not total 100%.


In [22]:
partial_rows = df.loc[
    partial,
    ["date", "gare_depart", "gare_arrivee", "nb_train_retard_arrivee"],
].copy()
partial_rows["cause_sum_pct"] = cause_sum[partial].round(2)
partial_rows["unclassified_trains"] = (
    (100 - cause_sum[partial]) * delayed[partial] / 100
).round().astype(int)

partial_rows.sort_values("cause_sum_pct").reset_index(drop=True)


,date,gare_depart,gare_arrivee,nb_train_retard_arrivee,cause_sum_pct,unclassified_trains
0,2026-06-01,PARIS MONTPARNASSE,LAVAL,65,84.62,10
1,2026-06-01,PARIS MONTPARNASSE,QUIMPER,46,86.96,6
2,2026-06-01,PARIS MONTPARNASSE,VANNES,56,87.50,7
3,2026-08-01,LILLE,LYON PART DIEU,41,97.56,1
4,2026-09-01,VALENCE ALIXAN TGV,PARIS LYON,49,97.96,1
5,2026-06-01,MARSEILLE ST CHARLES,LILLE,62,98.39,1
6,2026-09-01,LILLE,LYON PART DIEU,66,98.48,1
7,2026-08-01,MARSEILLE ST CHARLES,LILLE,69,98.55,1
8,2026-08-01,PARIS MONTPARNASSE,ST PIERRE DES CORPS,73,98.63,1
9,2026-06-01,LYON PART DIEU,LILLE,114,99.12,1


When focusing on the 10 cases where the sum of the causes does not add up exactly to 100%, we realize that there are in fact 7 cases where the sum of the causes is very close to 100%, with ultimately just one train whose delay cannot be explained. However, we have three other rows where the sum of the causes is below 90%, which increases the number of delayed trains whose cause cannot be explained. We therefore decide to keep the 7 rows with a cause sum close to 100% intact and document them, while for the other three, which are below 90%, we decide to replace the causes with NaN values. All other cases where there were delays and the cause percentages are equal to zero will be replaced with NaN values.

In [23]:
mask_missing_causes = (delayed > 0) & ~complete & (cause_sum < 90)
df.loc[mask_missing_causes, prct_causes] = np.nan

cause_sum_after = df[prct_causes].sum(axis=1)
known_causes = df[prct_causes].notna().any(axis=1)
kept_incomplete = (delayed > 0) & known_causes & ~complete

### 6.5 Average delays

The last checks target the average delays themselves.

In [24]:
# Case retard_moyen_*

mask_negative_delayed_means = (df["retard_moyen_depart"] < 0) | (df["retard_moyen_arrivee"] < 0)

print("Delayed-train means < 0:", int(mask_negative_delayed_means.sum()))
print("Delayed-train means at 0 with delayed trains:", int(mask_zero_delayed_means.sum()))

anomalies_delayed_means = df[mask_negative_delayed_means | mask_zero_delayed_means]
anomalies_delayed_means[
    [
        "date",
        "gare_depart",
        "gare_arrivee",
        "nb_train_retard_arrivee",
        "retard_moyen_arrivee",
        "retard_moyen_tous_trains_arrivee",
    ]
].head(12)

Delayed-train means < 0: 2
Delayed-train means at 0 with delayed trains: 38


,date,gare_depart,gare_arrivee,nb_train_retard_arrivee,retard_moyen_arrivee,retard_moyen_tous_trains_arrivee
2554,2019-09-01,BORDEAUX ST JEAN,TOURCOING,20,0.000000,3.100000
2836,2019-11-01,MONTPELLIER,PARIS LYON,44,-30.512500,-150.562114
2842,2019-11-01,NIMES,PARIS LYON,63,-40.109259,-151.291008
4518,2021-01-01,BARCELONA,PARIS LYON,2,0.000000,1.056667
4639,2021-02-01,BARCELONA,PARIS LYON,2,0.000000,1.196914
4760,2021-03-01,BARCELONA,PARIS LYON,3,0.000000,1.300556
4881,2021-04-01,BARCELONA,PARIS LYON,2,0.000000,1.343452
4966,2021-04-01,PARIS MONTPARNASSE,ST MALO,2,0.000000,0.635185
4999,2021-05-01,BARCELONA,PARIS LYON,3,0.000000,3.009444
5120,2021-06-01,BARCELONA,PARIS LYON,2,0.000000,3.055000


In [25]:
# Case retard_moyen_tous_*

mask_negative_all_train_means = (df["retard_moyen_tous_trains_depart"] < 0) | (df["retard_moyen_tous_trains_arrivee"] < 0)

for column in ["retard_moyen_tous_trains_depart", "retard_moyen_tous_trains_arrivee"]:
    print(column, "< 0:", int((df[column] < 0).sum()))

anomalies_all_train_means = df[mask_negative_all_train_means | mask_zero_all_train_means]
print("All-train means:", len(anomalies_all_train_means))

anomalies_all_train_means[
    [
        "date",
        "gare_depart",
        "gare_arrivee",
        "nb_train_depart_retard",
        "nb_train_retard_arrivee",
        "retard_moyen_tous_trains_depart",
        "retard_moyen_tous_trains_arrivee",
    ]
].sort_values("retard_moyen_tous_trains_arrivee").head(12)

retard_moyen_tous_trains_depart < 0: 169
retard_moyen_tous_trains_arrivee < 0: 122
All-train means: 277


,date,gare_depart,gare_arrivee,nb_train_depart_retard,nb_train_retard_arrivee,retard_moyen_tous_trains_depart,retard_moyen_tous_trains_arrivee
3026,2019-12-01,STRASBOURG,NANTES,6,0,1.244444,-472.638889
2944,2019-12-01,LYON PART DIEU,MONTPELLIER,39,15,10.288485,-173.076970
3212,2020-02-01,PARIS EST,FRANCFORT,111,18,-12.982278,-163.041983
2842,2019-11-01,NIMES,PARIS LYON,190,63,-112.262016,-151.291008
2836,2019-11-01,MONTPELLIER,PARIS LYON,189,44,4.277469,-150.562114
2719,2019-10-01,NIMES,PARIS LYON,203,70,-92.768307,-90.630840
2713,2019-10-01,MONTPELLIER,PARIS LYON,228,49,7.012749,-80.855113
2910,2019-11-01,VALENCE ALIXAN TGV,PARIS LYON,195,51,-69.838288,-71.528829
2787,2019-10-01,VALENCE ALIXAN TGV,PARIS LYON,221,68,-67.953740,-68.928084
2865,2019-11-01,PARIS LYON,MONTPELLIER,38,19,1.901355,-15.192547


The negative delayed-train means are impossible and 0 delayed-train means while delay counter is more than 1 is impossible neither. Without any further information about the data, it is impossible for us to determine the average delay. We therefore treat every anomaly all-train average as an aberration. We replace all of them with NaN, as well as the single zero all-train mean with delayed trains.

In [26]:
# Constraints on the mean delay of the trains late > 15 min.
mask_sup15_floor = (df["nb_train_retard_sup_15"] > 0) & (df["retard_moyen_trains_retard_sup15"] < 15)
mask_sup15_missing = (df["nb_train_retard_sup_15"] == 0) & (df["retard_moyen_trains_retard_sup15"] != 0)

print("Sup15 means < 15:", int((df[ "retard_moyen_trains_retard_sup15"] < 15).sum()))
print("without any train delayed > 15 min:", int(mask_sup15_missing.sum()))

anomalies_sup15 = df[mask_sup15_floor | mask_sup15_missing]
anomalies_sup15[
    [
        "date",
        "gare_depart",
        "gare_arrivee",
        "nb_train_retard_arrivee",
        "nb_train_retard_sup_15",
         "retard_moyen_trains_retard_sup15",
    ]
].head(12)

Sup15 means < 15: 2174
without any train delayed > 15 min: 16


,date,gare_depart,gare_arrivee,nb_train_retard_arrivee,nb_train_retard_sup_15,retard_moyen_trains_retard_sup15
0,2018-01-01,AIX EN PROVENCE TGV,PARIS LYON,37,37.0,4.417206
1,2018-01-01,ANGERS SAINT LAUD,PARIS MONTPARNASSE,77,35.0,6.072847
2,2018-01-01,ANGOULEME,PARIS MONTPARNASSE,56,33.0,6.464975
3,2018-01-01,ANNECY,PARIS LYON,38,38.0,8.552525
4,2018-01-01,ARRAS,PARIS NORD,64,16.0,5.351786
5,2018-01-01,AVIGNON TGV,PARIS LYON,88,62.0,5.797924
6,2018-01-01,BARCELONA,PARIS LYON,7,7.0,9.219048
7,2018-01-01,BELLEGARDE (AIN),PARIS LYON,53,42.0,7.531966
8,2018-01-01,BESANCON FRANCHE COMTE TGV,PARIS LYON,15,9.0,1.401320
9,2018-01-01,BORDEAUX ST JEAN,PARIS MONTPARNASSE,147,110.0,6.511118


 We are detecting trains with an average delay of more than 15 minutes, even though there are no train delays, and we see values below 15 minutes in this column. Both scenarios are impossible. In the last case, the true averages cannot be recovered, so we apply the minimum admissible value. The rows with trains delayed below 15 min are floored to 15, and the rows without any train delayed above 15 min should have a mean of 0, so they are replaced with 0.

In [27]:
# Delayed-train and all-train means: unrecoverable values -> NaN.
delayed_by_column = {
    "retard_moyen_depart": dep_delayed,
    "retard_moyen_tous_trains_depart": dep_delayed,
    "retard_moyen_arrivee": arr_delayed,
    "retard_moyen_tous_trains_arrivee": arr_delayed,
}

for column, delayed in delayed_by_column.items():
    mask_aberrant_mean = (df[column] < 0) | (delayed & (df[column] == 0))  # negative or zero while trains were delayed
    df.loc[mask_aberrant_mean, column] = np.nan

# Fix mean delay of the trains late below 15 min.
df.loc[mask_sup15_floor,  "retard_moyen_trains_retard_sup15"] = 15    
df.loc[mask_sup15_missing,  "retard_moyen_trains_retard_sup15"] = 0